Chapter 3 covers Time Series Analysis. In business analytics and risk modeling, time-series data usually arrives as granular event logs (individual transactions, sensor readings, system events).

The primary goal of Chapter 3 is learning how to transform point-in-time transactions into continuous, structured temporal aggregations (daily, monthly, quarterly) and perform relative period-over-period comparisons inside the database engine.

| Concept              | Problem Solved                                                     | Primary SQL Syntax                                                       |
| :------------------- | :----------------------------------------------------------------- | :----------------------------------------------------------------------- |
| **Date Truncation**  | Collapsing irregular timestamps into uniform daily/monthly buckets | `DATE_TRUNC('month', date)`                                              |
| **Moving Averages**  | Smoothing out seasonal noise and high-frequency volatility         | `AVG(val) OVER (ORDER BY date ROWS BETWEEN N PRECEDING AND CURRENT ROW)` |
| **MoM / YoY Deltas** | Measuring period-over-period growth or rate of change              | `LAG(val, 1) OVER (ORDER BY date)` / `LAG(val, 12)`                      |
| **Date Spines**      | Fixing missing dates/gaps in time-series sequences                 | `GENERATE_SERIES(start, end, interval) + LEFT JOIN`                      |


In [ ]:
-- Temporal Bucketing & Truncation

-- To group time-series data by time buckets (e.g., converting 2026-08-01 11:58:14 into 2026-08-01 or 2026-08), databases use date truncation functions.

-- Key Concept: DATE_TRUNC('month', date_column) chops off the trailing precision, snapping every timestamp to the beginning of its interval.

-- Why it matters: It gives you a clean key to execute GROUP BY operations over uniform temporal windows.

-- name: bucket_by_month
-- Snap every transaction date to the 1st of its month and aggregate sales

SELECT
    DATE_TRUNC('month', CAST(sales_month AS DATE)) AS sales_month_bucket,
    SUM(sales) AS total_monthly_sales
FROM retail_sales
GROUP BY 1
ORDER BY sales_month_bucket;

-- name: bucket_by_year
-- Snap every transaction date to Jan 1st of its year
SELECT
    DATE_TRUNC('year', CAST(sales_month AS DATE)) AS sales_year_bucket,
    SUM(sales) AS total_yearly_sales,
    AVG(sales) AS avg_monthly_sales
FROM retail_sales
GROUP BY 1
ORDER BY sales_year_bucket;

In [ ]:
-- Window Functions over Time (OVER (...))

-- Instead of collapsing rows with a standard GROUP BY, window functions compute metrics across a sliding frame while keeping all individual rows intact.

-- Syntax Structure: FUNCTION() OVER (PARTITION BY group_col ORDER BY time_col [RANGE/ROWS frame])

-- Core Application: Calculating moving averages, cumulative sums (running totals), and rolling volatility metrics directly on set partitions.

-- name: rolling_and_cumulative_sales
-- Calculates a 3-month moving average and a running total over time

SELECT
    CAST(sales_month AS DATE) AS sales_date,
    sales,

    -- 1. Running Total (Cumulative Sum from the start up to the current row)
    SUM(sales) OVER (
        ORDER BY CAST(sales_month AS DATE)
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total_sales,

    -- 2. Rolling 3-Month Moving Average (Current row + 2 previous rows)
    AVG(sales) OVER (
        ORDER BY CAST(sales_month AS DATE)
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rolling_3mo_avg_sales

FROM retail_sales
ORDER BY sales_date;


-- name: partitioned_rolling_sales
-- If your dataset has multiple categories/stores, windowing per partition
SELECT
    kind_of_business,
    CAST(sales_month AS DATE) AS sales_date,
    sales,

    -- Cumulative sales PER BUSINESS CATEGORY
    SUM(sales) OVER (
        PARTITION BY kind_of_business
        ORDER BY CAST(sales_month AS DATE)
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS category_running_total

FROM retail_sales
ORDER BY kind_of_business, sales_date;

In [ ]:
-- Lag and Lead Operations (Period-over-Period Deltas)

-- Comparing current metrics to past or future periods (e.g., month-over-month growth $M/M$ or year-over-year $Y/Y$) requires accessing adjacent rows without writing recursive self-joins.

-- LAG(column, offset): Reaches back $N$ rows in the partition order (e.g., prior month's sales).

-- LEAD(column, offset): Reaches forward $N$ rows.

-- name: mom_and_yoy_growth
-- Calculates Month-over-Month (MoM) and Year-over-Year (YoY) growth rates

WITH monthly_sales AS (
    SELECT
        DATE_TRUNC('month', CAST(sales_month AS DATE)) AS sales_month,
        SUM(sales) AS total_sales
    FROM retail_sales
    GROUP BY 1
)
SELECT
    sales_month,
    total_sales,

    -- 1. Grab previous month's sales (offset = 1)
    LAG(total_sales, 1) OVER (ORDER BY sales_month) AS prev_month_sales,

    -- Calculate MoM $ Change
    total_sales - LAG(total_sales, 1) OVER (ORDER BY sales_month) AS mom_dollar_change,

    -- Calculate MoM % Growth
    ROUND(
        (total_sales - LAG(total_sales, 1) OVER (ORDER BY sales_month))
        / LAG(total_sales, 1) OVER (ORDER BY sales_month) * 100, 2
    ) AS mom_pct_growth,

    -- 2. Grab sales from 12 months ago for YoY comparison (offset = 12)
    LAG(total_sales, 12) OVER (ORDER BY sales_month) AS same_month_prev_year_sales,

    -- Calculate YoY % Growth
    ROUND(
        (total_sales - LAG(total_sales, 12) OVER (ORDER BY sales_month))
        / LAG(total_sales, 12) OVER (ORDER BY sales_month) * 100, 2
    ) AS yoy_pct_growth

FROM monthly_sales
ORDER BY sales_month;


-- name: lead_next_period_forecast
-- Uses LEAD() to look ahead 1 month (useful for comparing actuals to next month's targets)

SELECT
    CAST(sales_month AS DATE) AS sales_date,
    sales AS current_month_sales,

    -- Reaches FORWARD 1 row
    LEAD(sales, 1) OVER (ORDER BY CAST(sales_month AS DATE)) AS next_month_sales
FROM retail_sales
ORDER BY sales_date;

In [ ]:
-- 4. Handling Missing Periods (Date Spines / Saffrons)

-- If a business makes zero sales on a given day, SQL's GROUP BY will completely drop that date from the query output. This creates discontinuous time series that break downstream quantitative models.

-- The Solution: Generating a continuous "date spine" (using GENERATE_SERIES() in DuckDB/Postgres) and LEFT JOIN-ing the event table against it, wrapping missing values in COALESCE(sales, 0).

-- name: filled_date_spine
-- Generates a daily date spine and fills missing sales days with 0

WITH date_spine AS (
    -- 1. Generate an unbroken daily sequence between min and max dates
    SELECT
        CAST(generate_series AS DATE) AS date_key
    FROM generate_series(
        DATE '2026-01-01',
        DATE '2026-01-10',
        INTERVAL '1 day'
    )
),
daily_sales AS (
    -- 2. Aggregate actual sales by date
    SELECT
        CAST(sales_month AS DATE) AS sales_date,
        SUM(sales) AS total_sales
    FROM retail_sales
    GROUP BY 1
)
-- 3. Join actuals to the date spine and replace NULLs with 0
SELECT
    s.date_key AS sales_date,
    COALESCE(d.total_sales, 0) AS total_sales,

    -- Now LAG(1) accurately references the previous actual day!
    LAG(COALESCE(d.total_sales, 0), 1) OVER (ORDER BY s.date_key) AS prev_day_sales

FROM date_spine s
LEFT JOIN daily_sales d
       ON s.date_key = d.sales_date
ORDER BY s.date_key;


-- name: dynamic_date_spine
-- Automatically determines start and end dates from the dataset

WITH bounds AS (
    SELECT
        MIN(CAST(sales_month AS DATE)) AS min_date,
        MAX(CAST(sales_month AS DATE)) AS max_date
    FROM retail_sales
),
date_spine AS (
    SELECT
        CAST(generate_series AS DATE) AS date_key
    FROM bounds, generate_series(bounds.min_date, bounds.max_date, INTERVAL '1 month')
)
SELECT
    s.date_key AS sales_month,
    COALESCE(SUM(r.sales), 0) AS total_sales
FROM date_spine s
LEFT JOIN retail_sales r
       ON s.date_key = CAST(r.sales_month AS DATE)
GROUP BY 1
ORDER BY sales_month;


In [ ]:
-- 5. Cumulative & Year-to-Date (YTD) Metrics

-- Tracking cumulative progress requires bounding window frames relative to calendar anchor dates.

-- YTD Calculation: SUM(sales) OVER (PARTITION BY YEAR(sales_month) ORDER BY sales_month)

-- name: ytd_sales_and_running_totals
-- Calculates YTD sales (resets every Jan 1) alongside an overall lifetime running total

WITH monthly_aggregated AS (
    SELECT
        DATE_TRUNC('month', CAST(sales_month AS DATE)) AS sales_month,
        EXTRACT(YEAR FROM CAST(sales_month AS DATE)) AS sales_year,
        SUM(sales) AS total_sales
    FROM retail_sales
    GROUP BY 1, 2
)
SELECT
    sales_month,
    total_sales,

    -- 1. YTD Total: PARTITION BY sales_year forces the sum to reset each year
    SUM(total_sales) OVER (
        PARTITION BY sales_year
        ORDER BY sales_month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS ytd_sales,

    -- 2. Lifetime Cumulative Total: NO PARTITION BY means it accumulates endlessly
    SUM(total_sales) OVER (
        ORDER BY sales_month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS lifetime_cumulative_sales

FROM monthly_aggregated
ORDER BY sales_month;

In [ ]:
import duckdb

conn = duckdb.connect(database=":memory:")

# Ingest local CSV dataset
conn.execute(
    "CREATE TABLE retail_sales AS SELECT * FROM read_csv_auto('us_retail_sales.csv');"
)

# Chapter 3 Execution: Moving Averages, Prior Period Comparisons, and YoY Deltas
ch3_analysis_df = conn.execute("""
WITH monthly_total AS (
    -- Step 1: Bucket and Aggregate Total Sales by Month
    SELECT
        CAST(sales_month AS DATE) AS sales_month,
        SUM(sales) AS total_sales
    FROM retail_sales
    WHERE kind_of_business = 'Retail and food services sales, total'
    GROUP BY 1
),
time_series_metrics AS (
    -- Step 2: Apply Window Functions for Rolling Windows & Lags
    SELECT
        sales_month,
        total_sales,

        -- Prior Month Sales (M/M)
        LAG(total_sales, 1) OVER (ORDER BY sales_month) AS prev_month_sales,

        -- Prior Year Sales (Y/Y)
        LAG(total_sales, 12) OVER (ORDER BY sales_month) AS prev_year_sales,

        -- 12-Month Moving Average
        AVG(total_sales) OVER (
            ORDER BY sales_month
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS rolling_12mo_avg
    FROM monthly_total
)
-- Step 3: Compute Relative Differences
SELECT
    sales_month,
    total_sales,
    rolling_12mo_avg,

    -- Month-over-Month % Change
    ROUND(((total_sales - prev_month_sales) / prev_month_sales) * 100, 2) AS mom_pct_change,

    -- Year-over-Year % Change
    ROUND(((total_sales - prev_year_sales) / prev_year_sales) * 100, 2) AS yoy_pct_change
FROM time_series_metrics
ORDER BY sales_month DESC
LIMIT 12;
""").df()

ch3_analysis_df

In [13]:
import duckdb

# 1. Connect and ingest local CSV
conn = duckdb.connect(database=":memory:")
conn.execute(
    "CREATE TABLE retail_sales AS SELECT * FROM read_csv_auto('us_retail_sales.csv');"
)

# 2. Read the full .sql file
with open("time series queries.sql", "r", encoding="utf-8") as f:
    raw_sql = f.read()

# 3. Clean and split queries by semicolon
queries = [q.strip() for q in raw_sql.split(";") if q.strip()]

print(f"Successfully loaded {len(queries)} queries from 'time series queries.sql'.")

Successfully loaded 34 queries from 'time series queries.sql'.


In [17]:
# Execute Query #1 (Chapter 3 First Example)
df = conn.execute(queries[0]).df()
df.head(10)

,sales_month,sales
0,1992-01-01,146376
1,1992-02-01,147079
2,1992-03-01,159336
3,1992-04-01,163669
4,1992-05-01,170068
5,1992-06-01,168663
6,1992-07-01,169890
7,1992-08-01,170364
8,1992-09-01,164617
9,1992-10-01,173655


In [ ]:
%%sql
-- Directly test CTEs, moving averages, or cohort dates from Chapter 3
SELECT
    sales_month,
    kind_of_business,
    sales,
    AVG(sales) OVER(
        PARTITION BY kind_of_business
        ORDER BY sales_month
        ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
    ) AS rolling_12mo_avg
FROM retail_sales
WHERE kind_of_business = 'Retail and food services sales, total'
LIMIT 12;

Running query in 'duckdb:///:memory:'

sales_month,kind_of_business,sales,rolling_12mo_avg
1992-01-01,"Retail and food services sales, total",146376,146376.0
1992-02-01,"Retail and food services sales, total",147079,146727.5
1992-03-01,"Retail and food services sales, total",159336,150930.33333333334
1992-04-01,"Retail and food services sales, total",163669,154115.0
1992-05-01,"Retail and food services sales, total",170068,157305.6
1992-06-01,"Retail and food services sales, total",168663,159198.5
1992-07-01,"Retail and food services sales, total",169890,160725.85714285713
1992-08-01,"Retail and food services sales, total",170364,161930.625
1992-09-01,"Retail and food services sales, total",164617,162229.11111111112
1992-10-01,"Retail and food services sales, total",173655,163371.7


In [ ]:
# putting queries in a dictionary

import re
import duckdb

conn = duckdb.connect(database=":memory:")
conn.execute(
    "CREATE TABLE retail_sales AS SELECT * FROM read_csv_auto('us_retail_sales.csv');"
)

# Read the file
with open("time series queries.sql", "r", encoding="utf-8") as f:
    raw_sql = f.read()

# Parse named queries using regex
queries = {}
raw_blocks = raw_sql.split(";")

for block in raw_blocks:
    if not block.strip():
        continue
    # Look for "-- name: query_name" at the start of the block
    match = re.search(r"--\s*name:\s*(\w+)", block)
    if match:
        query_name = match.group(1)
        queries[query_name] = block.strip()

print(f"Loaded queries: {list(queries.keys())}")

# Execute by descriptive name!
df = conn.execute(queries["monthly_sales_cast"]).df()
df.head(10)